# Light or Letdown? — Social Media Analysis of the **Ferrari Luce**
### Web and Social Media Search and Analysis — BSc AI, UniMiB

Self-contained Colab notebook for the Track-1 project (Network + Content + Visualization).
It writes the tested project code (`src/`) into Colab and runs the full pipeline end-to-end,
**including the Reddit scraping** (LAB 1–2).

**Runs with no credentials.** Reddit is scraped live via **Arctic-Shift → PullPush**
(no account, no key); Bluesky is included only if you add an App Password in §2. The full
Reddit pull takes ~10–15 min.

**How to use:** *Runtime → Change runtime type → GPU* (recommended — the transformer models
auto-detect and use it) → **Runtime → Run all**. Pipeline: collect → graph (L3) → communities
(L4) → sentiment/ABSA (L5) → sarcasm/stance/topics/language → NER (L6) → visualization →
download everything. See `MEGAPLAN.md`.

## 1. Install dependencies
Colab bundles torch/transformers/pandas/sklearn/networkx/nltk/matplotlib/spaCy. Install the rest.

In [ ]:
# Core pipeline deps. Colab already ships torch / transformers / pandas /
# scikit-learn / networkx / nltk / spaCy / matplotlib / seaborn / wordcloud / requests.
!pip install -q atproto praw vaderSentiment afinn NRCLex langdetect python-louvain python-dotenv

# BERTopic is an optional extension; if its install fails the pipeline falls
# back automatically to a scikit-learn LDA topic model (no action needed).
!pip install -q bertopic || echo "bertopic not installed -> sklearn LDA fallback will be used"

# spaCy English model for NER (LAB 6)
!python -m spacy download en_core_web_sm -q
print("dependencies installed.")

In [ ]:
import nltk
for pkg in ("punkt","punkt_tab","stopwords","wordnet","omw-1.4"):
    nltk.download(pkg, quiet=True)
print("NLTK data ready.")

## 2. Credentials — all OPTIONAL (the notebook runs with none)
- **Reddit:** leave blank → collected via **PullPush.io** (no account, no key, no OAuth).
- **Bluesky:** leave blank → Bluesky is **skipped**; to include it, add an *App Password* (bsky.app → Settings → App Passwords).

Nothing is stored. Best practice: use Colab **Secrets** (🔑 in the left sidebar) named `BLUESKY_HANDLE`, `BLUESKY_APP_PASSWORD`, `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET` — the next cell reads them automatically. Otherwise paste values into the cell. Either way **Runtime → Run all** completes unattended.

In [ ]:
import os

# ── Optional credentials ── leave blank to run with NO account ──────────────
# Reddit blank -> PullPush.io (no account).  Bluesky blank -> Bluesky skipped.
# Paste here (kept in memory only) OR, preferably, set them as Colab Secrets.
BLUESKY_HANDLE       = ""   # e.g. "you.bsky.social"
BLUESKY_APP_PASSWORD = ""   # App Password, NOT your main password
REDDIT_CLIENT_ID     = ""   # leave blank -> PullPush.io
REDDIT_CLIENT_SECRET = ""   # leave blank -> PullPush.io
REDDIT_USERNAME      = "student"   # only used to build a polite user-agent string

def _resolve(name, pasted):
    """Pasted value > Colab Secret > existing environment variable > ''."""
    if pasted:
        return pasted
    try:
        from google.colab import userdata          # only exists on Colab
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, "")

os.environ["BLUESKY_HANDLE"]       = _resolve("BLUESKY_HANDLE", BLUESKY_HANDLE)
os.environ["BLUESKY_APP_PASSWORD"] = _resolve("BLUESKY_APP_PASSWORD", BLUESKY_APP_PASSWORD)
os.environ["REDDIT_CLIENT_ID"]     = _resolve("REDDIT_CLIENT_ID", REDDIT_CLIENT_ID)
os.environ["REDDIT_CLIENT_SECRET"] = _resolve("REDDIT_CLIENT_SECRET", REDDIT_CLIENT_SECRET)
os.environ["REDDIT_USER_AGENT"]    = f"wsa-ferrari-luce/0.1 by u/{REDDIT_USERNAME or 'student'}"

print("Bluesky:", "ON" if os.environ["BLUESKY_HANDLE"] else "skipped (no creds)")
print("Reddit :", "PRAW (API)" if os.environ["REDDIT_CLIENT_ID"] else "PullPush.io (no account)")

## 3. Project code
This notebook **imports the tested package in `src/`** directly. If you opened the project
folder (uploaded to Colab or mounted from Drive) `src/` is already here; on a bare Colab the
cell below clones the repo to fetch it.

In [ ]:
import os, subprocess
# Make the src/ package available. If you're running inside the project folder,
# src/ is already here; otherwise clone the repo to fetch it.
if not os.path.isdir("src"):
    subprocess.run(
        "git clone --depth 1 https://github.com/akirykouski/wsa-ferrari-luce.git _wsa "
        "&& cp -r _wsa/src ./src && cp -r _wsa/report ./report 2>/dev/null",
        shell=True,
    )
if not os.path.isdir("src"):
    raise RuntimeError(
        "src/ not found. The repo is private, so either run this notebook from the "
        "project folder (upload it to Colab or mount Google Drive), or clone a repo "
        "you can access."
    )
for d in ("data/raw", "data/processed", "figures", "report"):
    os.makedirs(d, exist_ok=True)
print("project code ready:", os.path.isdir("src"))

## 4. Run the pipeline

In [ ]:
import pandas as pd
from IPython.display import Image, display
import glob

### 4.1 Collect the data (LAB 1–2)
Scrapes the real data **live**: the official Reddit API if you entered creds in §2,
otherwise **Arctic-Shift → PullPush** (no account, no key). Bluesky is included only if you
supplied an App Password. The full Reddit pull (~370 submissions + their comment trees)
takes **~10–15 min**; re-running fetches fresh posts.

In [ ]:
# ── Collect the data (LAB 1-2) ──────────────────────────────────────────
# Live scrape: Reddit API if creds were given in §2, else Arctic-Shift ->
# PullPush (no account). Bluesky only if an App Password was supplied.
from src.collect_bluesky import main as run_bluesky
from src.collect_reddit import main as run_reddit

try:
    run_bluesky()
except Exception as e:
    print("Bluesky collection skipped:", e)

run_reddit()

### 4.2 Network analysis — graph + centralities (LAB 3), communities (LAB 4)

In [ ]:
from src.build_graph import main as run_graph
from src.communities import main as run_comm
run_graph(); run_comm()
print(open("data/processed/graph_summary.txt").read())
print(open("data/processed/communities_summary.txt").read())
display(pd.read_csv("data/processed/nodes_centrality.csv").head(15))

### 4.3 Content — sentiment/emotion/ABSA (LAB 5) + enrichments (sarcasm RQ6, stance RQ7, topics, language RQ8)

In [ ]:
from src.content_sentiment import main as run_sent
from src.content_enrich import main as run_enrich
run_sent(); run_enrich()
display(pd.read_csv("data/processed/aspect_sentiment.csv"))
display(pd.read_csv("data/processed/language_sentiment.csv"))

### 4.4 NER + entity-level sentiment (LAB 6)

In [ ]:
from src.ner_entities import main as run_ner
run_ner()
display(pd.read_csv("data/processed/entity_frequency.csv").head(20))
display(pd.read_csv("data/processed/entity_sentiment.csv").head(20))

### 4.5 Visualizations
All charts are saved to `figures/` as PNGs, including the community-coloured interaction network.

In [ ]:
from src.viz import main as run_viz
run_viz()
for f in sorted(glob.glob("figures/*.png")):
    print(f); display(Image(f))

### 4.6 Download everything
One Colab run produces the complete result set from the bundled data. This packs all
tables (`data/processed/`) and charts (`figures/`, including the interactive
`community_network.html`) into a single zip and downloads it — so you leave with everything.

In [ ]:
import os, zipfile
ZIP = "WSA_FerrariLuce_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for root in ("data/processed", "figures"):
        for dp, _, files in os.walk(root):
            for f in files:
                if f == ".gitkeep":
                    continue
                z.write(os.path.join(dp, f))
print(f"bundled all results -> {ZIP} ({os.path.getsize(ZIP) // 1024} KB)")

# On Colab this triggers a browser download of the full bundle.
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    print("(not on Colab — grab the zip from the file browser on the left)")

# Optional: copy everything to Google Drive instead of downloading.
# from google.colab import drive; drive.mount("/content/drive")
# !mkdir -p "/content/drive/MyDrive/WSA_FerrariLuce" && cp -r data figures "/content/drive/MyDrive/WSA_FerrariLuce/"